# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR⁲) Exploration with `mlcroissant`

This notebook demonstrates how to reproducibly load and analyze a biomedical Croissant dataset using the `mlcroissant` library. We will walk through retrieving the schema and records, exploring its record sets and fields by `@id`, and performing preliminary processing and visualization.

### Dataset Source

The dataset is provided as a Croissant schema:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

The data contains comprehensive clinicopathological and biomarker variables (including MSI/MMR status, anatomical distribution, comorbidities) for 77 cancer survivors with second primary colorectal cancer.

In [ ]:
# Ensure mlcroissant is available
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Print dataset name and description from metadata
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview

List the available record sets, fields, and their `@id` values.

We will explore the dataset schema and print out each record set by their `@id` as well as the fields and columns contained within.

In [ ]:
# List all record sets by @id
record_sets = dataset.metadata.record_sets
print(f"Found {len(record_sets)} record sets.\n")

for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name if hasattr(rs, 'name') else ''}")
    # List fields by @id
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for field in rs.fields:
            print(f"    Field @id: {field.id} | name: {getattr(field, 'name', '')}")
            # List columns in this field (if present)
            if hasattr(field, 'columns') and field.columns:
                for column in field.columns:
                    print(f"      Column @id: {column.id} | name: {getattr(column, 'name', '')}")
    print()

## 3. Data Extraction

Now, let's load data from the main record set(s) into pandas DataFrame(s) for analysis. All selection will use the record set and field `@id` values found above.

In [ ]:
# Gather all record set @id
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Extracting records for RecordSet @id: {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    if not df.empty:
        print(f"  Columns in DataFrame for {record_set_id}: {df.columns.tolist()}\n")
    else:
        print(f"  No records found for {record_set_id}.\n")

# Choose the main record set as example (use first one for demo)
main_record_set_id = record_set_ids[0]

print(f"Preview from RecordSet @id: {main_record_set_id}")
display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's process the main DataFrame:
- We'll filter on a numeric column (for example, age or diagnosis interval).
- We'll normalize the numeric column (z-score normalization).
- We'll group by a categorical variable (such as MSI status or anatomical site), where available.

All reference is by the field or column `@id` found earlier.

In [ ]:
# Adjust the field @id as needed for your dataset
# For this biomedical dataset, let's assume one numeric field is 'age' (replace with actual @id if different)

main_df = dataframes[main_record_set_id].copy()

# Inspect column names to identify suitable numeric and group fields
print("Columns available in the main DataFrame:")
print(main_df.columns.tolist())

# Example: Suppose '@id' for age column is 'age' (replace as found in your schema)
numeric_field_id = None
group_field_id = None
# Try common variations, replace with exact values as needed
for col in main_df.columns:
    if 'age' in col.lower():
        numeric_field_id = col
    if any(x in col.lower() for x in ['msi', 'anatom', 'mmr']):
        group_field_id = col

if numeric_field_id is not None:
    threshold = 60  # filter e.g. age > 60 years
    filtered_df = main_df[main_df[numeric_field_id].astype(float) > threshold].copy()
    print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()
    ) / filtered_df[numeric_field_id].astype(float).std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field if available
    if group_field_id is not None:
        print(f"\nGrouped statistics by {group_field_id}:")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        display(grouped_df)
else:
    print("No suitable numeric field (e.g. Age) found; please check schema and columns.")

## 5. Visualization

To better understand the distribution and relationships in the data, let's plot numeric and categorical variables (if such fields exist).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Plot the distribution of the numeric field (e.g. Age)
if numeric_field_id is not None:
    plt.figure(figsize=(7,4))
    sns.histplot(main_df[numeric_field_id].astype(float), bins=12, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# Example: Boxplot of numeric field grouped by a categorical field, e.g., MSI/MMR status or anatomical location
if group_field_id is not None and numeric_field_id is not None:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=main_df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=30, ha='right')
    plt.show()


## 6. Conclusion

* Using the `mlcroissant` library, we loaded and overviewed a Croissant-described FAIR biomedical dataset, referencing all entities by their `@id`.
* We programmatically explored available record sets, fields, and columns, and demonstrated record extraction by `@id`.
* Preliminary EDA revealed the numeric field's distribution (e.g., Age) and allowed grouping by categorical variables (e.g., MSI status).

For further analysis, consult the full schema documentation to use precise field `@id`s for robust, reproducible research workflows.